In [1]:
# Percobaan 5 - Simple CatBoost Full Ensemble + Jalur A

Versi lanjutan dari `simple_catboost_baseline.ipynb`.

Target notebook ini:
- tetap memakai preprocessing simple yang sudah menghasilkan LB sekitar 3.14,
- train beberapa variasi CatBoost yang stabil,
- optional menambahkan LightGBM/XGBoost kalau library tersedia,
- memilih blend weight berdasarkan validation AW-MAE,
- menambahkan Jalur A: outcome-aware post-processing + score calibration,
- generate submission ensemble Jalur A.

Catatan: kalau waktu mepet, jalankan dulu CatBoost ensemble + Jalur A. Model opsional otomatis di-skip kalau package tidak tersedia.


In [2]:
BASE_PATH = Path.home() / 'Downloads' / 'Gammafest'
DATA_PATH = BASE_PATH / 'dataset'
OUTPUT_DIR = BASE_PATH / 'experiments' / 'percobaan 5 - simple baseline'

TRAIN_PATH = DATA_PATH / 'train.csv'
TEST_PATH = DATA_PATH / 'test.csv'
SAMPLE_PATH = DATA_PATH / 'sample submission.csv'
SUBMISSION_PATH = OUTPUT_DIR / 'submission_full_ensemble_jalur_a.csv'
SUBMISSION_ROUNDCLIP_PATH = OUTPUT_DIR / 'submission_full_ensemble_roundclip.csv'
OOF_REPORT_PATH = OUTPUT_DIR / 'ensemble_validation_report.csv'
POSTPROCESS_REPORT_PATH = OUTPUT_DIR / 'jalur_a_postprocess_report.csv'

RANDOM_STATE = 42
VALID_FRAC = 0.20
TASK_TYPE = 'GPU'  # ganti ke 'GPU' kalau ingin coba CatBoost GPU
MAX_SCORE = 6
N_RANDOM_BLENDS = 2500

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)

Output: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline


In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print('train:', train_raw.shape)
print('test :', test_raw.shape)
print('sample:', sample.shape)
display(train_raw.head(2))
display(test_raw.head(2))

train: (78772, 47)
test : (42422, 20)
sample: (42422, 3)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,team_points_last5,opp_points_last5,points_last5_diff,team_gd_last5,opp_gd_last5,gd_last5_diff,h2h_points_last5,h2h_gd_last5,days_since_last_match_team,days_since_last_match_opp,team_points_last10,opp_points_last10,team_avg_goals_last5,team_avg_conceded_last5,opp_avg_goals_last5,opp_avg_conceded_last5,team_win_rate_last10,opp_win_rate_last10,elo_team,elo_opponent,rank_team,rank_opponent,rank_diff,rank_missing_team,rank_missing_opp,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,0,0,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,0.0,0.0,NaN,0.0,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,92409.0,1283330.0,12189.095160,9197.026972,-9999.0,0.000000,1751.895724,25.790169
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,1283330.0,92409.0,9197.026972,12189.095160,-9999.0,1751.895724,0.000000,25.790169


In [4]:
CAT_COLS = [
    'gender', 'team', 'opponent', 'tournament', 'venue_country',
    'confederation_team', 'confederation_opp',
]

NUM_COLS = [
    'is_home', 'neutral',
    'population_team', 'population_opp',
    'gdp_per_capita_team', 'gdp_per_capita_opp',
    'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue',
]

DATE_FEATURES = ['year', 'month', 'dayofweek']
TARGET_COLS = ['team_goals', 'opp_goals']


def add_date_features(df):
    df = df.copy()
    dt = pd.to_datetime(df['date'], errors='coerce')
    df['year'] = dt.dt.year
    df['month'] = dt.dt.month
    df['dayofweek'] = dt.dt.dayofweek
    return df


def prepare_features(train_df, test_df, cat_cols, num_cols):
    train = add_date_features(train_df)
    test = add_date_features(test_df)
    numeric_base = num_cols + DATE_FEATURES

    for df in [train, test]:
        for col in numeric_base:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.loc[df[col] == -9999, col] = np.nan

    missing_flag_cols = []
    for col in numeric_base:
        flag_col = f'{col}_missing'
        train[flag_col] = train[col].isna().astype(int)
        test[flag_col] = test[col].isna().astype(int)
        missing_flag_cols.append(flag_col)

    medians = train[numeric_base].median(numeric_only=True)
    for col in numeric_base:
        fill_value = medians[col]
        if pd.isna(fill_value):
            fill_value = 0
        train[col] = train[col].fillna(fill_value)
        test[col] = test[col].fillna(fill_value)

    for col in cat_cols:
        train[col] = train[col].fillna('Unknown').astype(str)
        test[col] = test[col].fillna('Unknown').astype(str)

    feature_cols = cat_cols + numeric_base + missing_flag_cols
    return train, test, feature_cols, missing_flag_cols, medians

train_prep, test_prep, FEATURE_COLS, MISSING_FLAG_COLS, TRAIN_MEDIANS = prepare_features(train_raw, test_raw, CAT_COLS, NUM_COLS)
cat_feature_indices = [FEATURE_COLS.index(c) for c in CAT_COLS]

print('Total features:', len(FEATURE_COLS))
print('Missing after preprocessing:', train_prep[FEATURE_COLS].isna().sum().sum(), test_prep[FEATURE_COLS].isna().sum().sum())
print(FEATURE_COLS)

Total features: 33
Missing after preprocessing: 0 0
['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp', 'is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue', 'year', 'month', 'dayofweek', 'is_home_missing', 'neutral_missing', 'population_team_missing', 'population_opp_missing', 'gdp_per_capita_team_missing', 'gdp_per_capita_opp_missing', 'altitude_venue_missing', 'distance_travel_team_missing', 'distance_travel_opp_missing', 'temperature_venue_missing', 'year_missing', 'month_missing', 'dayofweek_missing']


In [5]:
train_prep['date_dt'] = pd.to_datetime(train_prep['date'], errors='coerce')
train_sorted = train_prep.sort_values('date_dt').reset_index(drop=True)

split_idx = int(len(train_sorted) * (1 - VALID_FRAC))
tr_df = train_sorted.iloc[:split_idx].copy()
val_df = train_sorted.iloc[split_idx:].copy()

X_tr = tr_df[FEATURE_COLS]
X_val = val_df[FEATURE_COLS]
y_tr_team = tr_df['team_goals']
y_tr_opp = tr_df['opp_goals']
y_val_team = val_df['team_goals']
y_val_opp = val_df['opp_goals']

X_full = train_prep[FEATURE_COLS]
y_full_team = train_prep['team_goals']
y_full_opp = train_prep['opp_goals']
X_test = test_prep[FEATURE_COLS]

print('Train fold:', tr_df.shape, tr_df['date_dt'].min(), '->', tr_df['date_dt'].max())
print('Valid fold:', val_df.shape, val_df['date_dt'].min(), '->', val_df['date_dt'].max())

Train fold: (63017, 64) 1872-11-30 00:00:00 -> 2005-02-01 00:00:00
Valid fold: (15755, 64) 2005-02-01 00:00:00 -> 2011-08-04 00:00:00


In [6]:
TOURNAMENT_WEIGHTS = {
    'FIFA World Cup': 2.00,
    'AFC Championship': 1.80,
    'AFC Asian Cup': 1.80,
    'UEFA Euro': 1.80,
    'Copa America': 1.80,
    'Copa Am?rica': 1.80,
    'Africa Cup of Nations': 1.80,
    'African Cup of Nations': 1.80,
    'Gold Cup': 1.75,
    'CONCACAF Gold Cup': 1.75,
    'FIFA World Cup qualification': 1.50,
    'UEFA Euro qualification': 1.40,
    'AFC Asian Cup qualification': 1.40,
    'Friendly': 0.96,
}
DEFAULT_TOURNAMENT_WEIGHT = 1.20


def outcome_label(team_goals, opp_goals):
    diff = np.asarray(team_goals) - np.asarray(opp_goals)
    return np.where(diff > 0, 1, np.where(diff < 0, -1, 0))


def postprocess_round_clip(team_pred, opp_pred, max_score=6):
    team = np.rint(team_pred).astype(int)
    opp = np.rint(opp_pred).astype(int)
    team = np.clip(team, 0, max_score)
    opp = np.clip(opp, 0, max_score)
    return team, opp


def compute_awmae(df_true, team_pred, opp_pred):
    true_team = df_true['team_goals'].to_numpy()
    true_opp = df_true['opp_goals'].to_numpy()
    pred_team = np.asarray(team_pred)
    pred_opp = np.asarray(opp_pred)

    mae = (np.abs(true_team - pred_team) + np.abs(true_opp - pred_opp)) / 2
    exact = ((true_team == pred_team) & (true_opp == pred_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(pred_team, pred_opp)).astype(int)
    gd = ((true_team - true_opp) == (pred_team - pred_opp)).astype(int)

    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    loss = ((mae + penalty) * multiplier) ** 1.5
    weights = df_true['tournament'].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT).to_numpy()

    score = np.sum(loss * weights) / np.sum(weights)
    diag = {
        'AW-MAE': score,
        'MAE_raw_component': mae.mean(),
        'exact_acc': exact.mean(),
        'outcome_acc': outcome.mean(),
        'goal_diff_acc': gd.mean(),
    }
    return score, diag


def evaluate_predictions(name, df_true, team_raw, opp_raw, max_score=6):
    team_int, opp_int = postprocess_round_clip(team_raw, opp_raw, max_score=max_score)
    score, diag = compute_awmae(df_true, team_int, opp_int)
    out = {'model': name, **diag}
    return out, team_int, opp_int

In [7]:
CATBOOST_CONFIGS = [
    {
        'name': 'cat_mae_d7_seed42',
        'params': dict(loss_function='MAE', eval_metric='MAE', iterations=1400, learning_rate=0.045, depth=7, l2_leaf_reg=6, random_strength=1.0, bagging_temperature=0.4, random_seed=42),
    },
    {
        'name': 'cat_mae_d6_seed7',
        'params': dict(loss_function='MAE', eval_metric='MAE', iterations=1600, learning_rate=0.040, depth=6, l2_leaf_reg=8, random_strength=1.5, bagging_temperature=0.8, random_seed=7),
    },
    {
        'name': 'cat_mae_d8_seed99',
        'params': dict(loss_function='MAE', eval_metric='MAE', iterations=1200, learning_rate=0.035, depth=8, l2_leaf_reg=10, random_strength=0.8, bagging_temperature=0.2, random_seed=99),
    },
    {
        'name': 'cat_rmse_d7_seed123',
        'params': dict(loss_function='RMSE', eval_metric='MAE', iterations=1400, learning_rate=0.040, depth=7, l2_leaf_reg=7, random_strength=1.2, bagging_temperature=0.6, random_seed=123),
    },
]

BASE_CAT_PARAMS = dict(
    task_type=TASK_TYPE,
    verbose=150,
    allow_writing_files=False,
)

print('CatBoost configs:', [c['name'] for c in CATBOOST_CONFIGS])

CatBoost configs: ['cat_mae_d7_seed42', 'cat_mae_d6_seed7', 'cat_mae_d8_seed99', 'cat_rmse_d7_seed123']


In [9]:
val_pred_bank = {}
model_reports = []
trained_val_models = {}

start = time.time()
for cfg in CATBOOST_CONFIGS:
    name = cfg['name']
    params = {**BASE_CAT_PARAMS, **cfg['params']}
    print('' + '=' * 80)
    print('Training', name)

    model_t = CatBoostRegressor(**params)
    model_o = CatBoostRegressor(**params)

    model_t.fit(
        X_tr, y_tr_team,
        cat_features=cat_feature_indices,
        eval_set=(X_val, y_val_team),
        use_best_model=True,
        early_stopping_rounds=140,
    )
    model_o.fit(
        X_tr, y_tr_opp,
        cat_features=cat_feature_indices,
        eval_set=(X_val, y_val_opp),
        use_best_model=True,
        early_stopping_rounds=140,
    )

    pred_t = np.clip(model_t.predict(X_val), 0, None)
    pred_o = np.clip(model_o.predict(X_val), 0, None)
    val_pred_bank[name] = (pred_t, pred_o)
    trained_val_models[name] = (model_t, model_o)

    report, _, _ = evaluate_predictions(name, val_df, pred_t, pred_o, max_score=MAX_SCORE)
    report['best_iter_team'] = model_t.best_iteration_
    report['best_iter_opp'] = model_o.best_iteration_
    model_reports.append(report)
    print(report)

print('Done in minutes:', (time.time() - start) / 60)
reports_df = pd.DataFrame(model_reports).sort_values('AW-MAE')
display(reports_df)

Training cat_mae_d7_seed42


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1751876	test: 1.1332264	best: 1.1332264 (0)	total: 65.3ms	remaining: 1m 31s
150:	learn: 1.1064068	test: 1.0825722	best: 1.0825722 (150)	total: 2.94s	remaining: 24.3s
300:	learn: 1.0782423	test: 1.0572911	best: 1.0572911 (300)	total: 5.76s	remaining: 21.1s
450:	learn: 1.0618372	test: 1.0471889	best: 1.0471889 (450)	total: 8.72s	remaining: 18.4s
600:	learn: 1.0506986	test: 1.0415529	best: 1.0415529 (600)	total: 11.7s	remaining: 15.5s
750:	learn: 1.0417668	test: 1.0381832	best: 1.0381820 (749)	total: 14.5s	remaining: 12.6s
900:	learn: 1.0348297	test: 1.0359433	best: 1.0359409 (899)	total: 17.4s	remaining: 9.64s
1050:	learn: 1.0284114	test: 1.0342356	best: 1.0342019 (1043)	total: 20.2s	remaining: 6.71s
1200:	learn: 1.0230255	test: 1.0331410	best: 1.0331222 (1178)	total: 23.3s	remaining: 3.86s
1350:	learn: 1.0183112	test: 1.0324412	best: 1.0324412 (1350)	total: 26.1s	remaining: 947ms
1399:	learn: 1.0170784	test: 1.0322217	best: 1.0322217 (1399)	total: 27.1s	remaining: 0us
bestTe

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1752145	test: 1.1330483	best: 1.1330483 (0)	total: 16.1ms	remaining: 22.5s
150:	learn: 1.1079000	test: 1.0836114	best: 1.0836114 (150)	total: 2.82s	remaining: 23.4s
300:	learn: 1.0804601	test: 1.0587454	best: 1.0587454 (300)	total: 5.66s	remaining: 20.7s
450:	learn: 1.0631852	test: 1.0462223	best: 1.0462223 (450)	total: 8.7s	remaining: 18.3s
600:	learn: 1.0519841	test: 1.0419910	best: 1.0419910 (600)	total: 12.1s	remaining: 16.1s
750:	learn: 1.0434028	test: 1.0394730	best: 1.0394730 (750)	total: 15.2s	remaining: 13.1s
900:	learn: 1.0362136	test: 1.0373105	best: 1.0373105 (900)	total: 17.9s	remaining: 9.94s
1050:	learn: 1.0296132	test: 1.0351920	best: 1.0351910 (1049)	total: 20.7s	remaining: 6.88s
1200:	learn: 1.0246322	test: 1.0338039	best: 1.0338039 (1200)	total: 23.5s	remaining: 3.89s
1350:	learn: 1.0204095	test: 1.0332394	best: 1.0332052 (1341)	total: 26.3s	remaining: 953ms
1399:	learn: 1.0189443	test: 1.0328886	best: 1.0328682 (1397)	total: 27.1s	remaining: 0us
bestTest

...

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2780720	test: 1.2692681	best: 1.2692681 (0)	total: 15.7ms	remaining: 22s
150:	learn: 1.0792931	test: 1.0853545	best: 1.0853545 (150)	total: 2.84s	remaining: 23.5s
300:	learn: 1.0581685	test: 1.0794823	best: 1.0794823 (300)	total: 5.85s	remaining: 21.4s
450:	learn: 1.0443675	test: 1.0767599	best: 1.0764354 (449)	total: 8.72s	remaining: 18.4s
600:	learn: 1.0333718	test: 1.0732621	best: 1.0731862 (594)	total: 11.5s	remaining: 15.3s
bestTest = 1.073186216
bestIteration = 594
Shrink model to first 595 iterations.


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2777702	test: 1.2691322	best: 1.2691322 (0)	total: 16.4ms	remaining: 22.9s
150:	learn: 1.0785460	test: 1.0887243	best: 1.0887063 (149)	total: 2.83s	remaining: 23.4s
300:	learn: 1.0566684	test: 1.0856831	best: 1.0856831 (300)	total: 5.85s	remaining: 21.4s
450:	learn: 1.0435706	test: 1.0815186	best: 1.0815069 (449)	total: 9.06s	remaining: 19.1s
600:	learn: 1.0329845	test: 1.0802798	best: 1.0799735 (582)	total: 12.2s	remaining: 16.2s
750:	learn: 1.0241713	test: 1.0778681	best: 1.0777242 (722)	total: 15.1s	remaining: 13s
900:	learn: 1.0154159	test: 1.0773912	best: 1.0767882 (830)	total: 18s	remaining: 9.98s
bestTest = 1.076788247
bestIteration = 830
Shrink model to first 831 iterations.
{'model': 'cat_rmse_d7_seed123', 'AW-MAE': np.float64(3.1494843141016386), 'MAE_raw_component': np.float64(1.0447159631862901), 'exact_acc': np.float64(0.09343065693430656), 'outcome_acc': np.float64(0.5484608060933037), 'goal_diff_acc': np.float64(0.2344652491272612), 'best_iter_team': 594, 'be

,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
3,cat_rmse_d7_seed123,3.149484,1.044716,0.093431,0.548461,0.234465,594,830
0,cat_mae_d7_seed42,3.198458,1.013170,0.103078,0.488036,0.234973,1399,1397
1,cat_mae_d6_seed7,3.217227,1.017010,0.102380,0.484735,0.235036,1596,1599
2,cat_mae_d8_seed99,3.229840,1.014948,0.103332,0.478705,0.235100,1197,1199


In [12]:
def make_encoded_data():
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    Xtr_enc = X_tr.copy()
    Xval_enc = X_val.copy()
    Xfull_enc = X_full.copy()
    Xtest_enc = X_test.copy()

    encoder.fit(Xtr_enc[CAT_COLS])
    for X in [Xtr_enc, Xval_enc, Xfull_enc, Xtest_enc]:
        X[CAT_COLS] = encoder.transform(X[CAT_COLS]).astype('float32')
    return Xtr_enc, Xval_enc, Xfull_enc, Xtest_enc

Xtr_enc, Xval_enc, Xfull_enc, Xtest_enc = make_encoded_data()

# LightGBM optional
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception as e:
    HAS_LGBM = False
    print('LightGBM skipped:', repr(e))

if HAS_LGBM:
    lgbm_configs = [
        ('lgbm_mae_seed42', dict(objective='regression_l1', n_estimators=1400, learning_rate=0.035, num_leaves=63, subsample=0.85, colsample_bytree=0.85, reg_lambda=6.0, random_state=42)),
        ('lgbm_mae_seed99', dict(objective='regression_l1', n_estimators=1200, learning_rate=0.040, num_leaves=47, subsample=0.90, colsample_bytree=0.80, reg_lambda=9.0, random_state=99)),
    ]
    for name, params in lgbm_configs:
        print('Training', name)
        mt = LGBMRegressor(**params, verbosity=-1)
        mo = LGBMRegressor(**params, verbosity=-1)
        mt.fit(Xtr_enc, y_tr_team)
        mo.fit(Xtr_enc, y_tr_opp)
        pred_t = np.clip(mt.predict(Xval_enc), 0, None)
        pred_o = np.clip(mo.predict(Xval_enc), 0, None)
        val_pred_bank[name] = (pred_t, pred_o)
        trained_val_models[name] = (mt, mo)
        report, _, _ = evaluate_predictions(name, val_df, pred_t, pred_o, max_score=MAX_SCORE)
        model_reports.append(report)
        print(report)

# XGBoost optional
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print('XGBoost skipped:', repr(e))

if HAS_XGB:
    xgb_configs = [
        ('xgb_mae_seed42', dict(n_estimators=1200, learning_rate=0.035, max_depth=6, min_child_weight=8, subsample=0.85, colsample_bytree=0.85, reg_lambda=8.0, random_state=42)),
        ('xgb_mae_seed123', dict(n_estimators=1000, learning_rate=0.040, max_depth=5, min_child_weight=10, subsample=0.90, colsample_bytree=0.80, reg_lambda=10.0, random_state=123)),
    ]
    for name, params in xgb_configs:
        print('Training', name)
        base = dict(objective='reg:absoluteerror', tree_method='hist', n_jobs=-1)
        mt = XGBRegressor(**base, **params)
        mo = XGBRegressor(**base, **params)
        mt.fit(Xtr_enc, y_tr_team, verbose=False)
        mo.fit(Xtr_enc, y_tr_opp, verbose=False)
        pred_t = np.clip(mt.predict(Xval_enc), 0, None)
        pred_o = np.clip(mo.predict(Xval_enc), 0, None)
        val_pred_bank[name] = (pred_t, pred_o)
        trained_val_models[name] = (mt, mo)
        report, _, _ = evaluate_predictions(name, val_df, pred_t, pred_o, max_score=MAX_SCORE)
        model_reports.append(report)
        print(report)

reports_df = pd.DataFrame(model_reports).sort_values('AW-MAE')
reports_df.to_csv(OOF_REPORT_PATH, index=False)
display(reports_df)
print('Saved report:', OOF_REPORT_PATH)

Training lgbm_mae_seed42
{'model': 'lgbm_mae_seed42', 'AW-MAE': np.float64(3.307611480335848), 'MAE_raw_component': np.float64(1.0377340526816883), 'exact_acc': np.float64(0.10384005077753095), 'outcome_acc': np.float64(0.48721040939384325), 'goal_diff_acc': np.float64(0.2365598222786417)}
Training lgbm_mae_seed99
{'model': 'lgbm_mae_seed99', 'AW-MAE': np.float64(3.309607151506807), 'MAE_raw_component': np.float64(1.0332910187242146), 'exact_acc': np.float64(0.10580768010155506), 'outcome_acc': np.float64(0.47940336401142497), 'goal_diff_acc': np.float64(0.23814662012059665)}
Training xgb_mae_seed42
{'model': 'xgb_mae_seed42', 'AW-MAE': np.float64(3.353179672280502), 'MAE_raw_component': np.float64(1.0403681370993336), 'exact_acc': np.float64(0.10403046651856554), 'outcome_acc': np.float64(0.4755950491907331), 'goal_diff_acc': np.float64(0.23541732783243416)}
Training xgb_mae_seed123
{'model': 'xgb_mae_seed123', 'AW-MAE': np.float64(3.401745096445464), 'MAE_raw_component': np.float64(1

,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
3,cat_rmse_d7_seed123,3.149484,1.044716,0.093431,0.548461,0.234465,594.0,830.0
0,cat_mae_d7_seed42,3.198458,1.013170,0.103078,0.488036,0.234973,1399.0,1397.0
1,cat_mae_d6_seed7,3.217227,1.017010,0.102380,0.484735,0.235036,1596.0,1599.0
2,cat_mae_d8_seed99,3.229840,1.014948,0.103332,0.478705,0.235100,1197.0,1199.0
4,lgbm_mae_seed42,3.307611,1.037734,0.103840,0.487210,0.236560,NaN,NaN
5,lgbm_mae_seed99,3.309607,1.033291,0.105808,0.479403,0.238147,NaN,NaN
6,xgb_mae_seed42,3.353180,1.040368,0.104030,0.475595,0.235417,NaN,NaN
7,xgb_mae_seed123,3.401745,1.043637,0.104221,0.461949,0.234275,NaN,NaN


Saved report: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 5 - simple baseline\ensemble_validation_report.csv


In [13]:
model_names = list(val_pred_bank.keys())
team_matrix = np.vstack([val_pred_bank[name][0] for name in model_names])
opp_matrix = np.vstack([val_pred_bank[name][1] for name in model_names])

print('Models in bank:', model_names)
print('Prediction matrix:', team_matrix.shape, opp_matrix.shape)


def score_blend(weights, max_score=MAX_SCORE):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    pred_t = np.average(team_matrix, axis=0, weights=weights)
    pred_o = np.average(opp_matrix, axis=0, weights=weights)
    pred_t_int, pred_o_int = postprocess_round_clip(pred_t, pred_o, max_score=max_score)
    score, diag = compute_awmae(val_df, pred_t_int, pred_o_int)
    return score, diag, pred_t_int, pred_o_int

blend_rows = []
best = {'score': np.inf, 'weights': None, 'name': None, 'diag': None}

# Single models
for i, name in enumerate(model_names):
    w = np.zeros(len(model_names))
    w[i] = 1.0
    score, diag, _, _ = score_blend(w)
    blend_rows.append({'blend': f'single_{name}', 'AW-MAE': score, **{f'w_{n}': w[j] for j, n in enumerate(model_names)}})
    if score < best['score']:
        best = {'score': score, 'weights': w.copy(), 'name': f'single_{name}', 'diag': diag}

# Equal average
w = np.ones(len(model_names)) / len(model_names)
score, diag, _, _ = score_blend(w)
blend_rows.append({'blend': 'equal_average', 'AW-MAE': score, **{f'w_{n}': w[j] for j, n in enumerate(model_names)}})
if score < best['score']:
    best = {'score': score, 'weights': w.copy(), 'name': 'equal_average', 'diag': diag}

# Inverse validation score weights
single_scores = []
for i, name in enumerate(model_names):
    w_single = np.zeros(len(model_names)); w_single[i] = 1
    s, _, _, _ = score_blend(w_single)
    single_scores.append(s)
w = 1 / np.maximum(np.array(single_scores), 1e-9)
w = w / w.sum()
score, diag, _, _ = score_blend(w)
blend_rows.append({'blend': 'inverse_awmae', 'AW-MAE': score, **{f'w_{n}': w[j] for j, n in enumerate(model_names)}})
if score < best['score']:
    best = {'score': score, 'weights': w.copy(), 'name': 'inverse_awmae', 'diag': diag}

# Random Dirichlet search
rng = np.random.default_rng(RANDOM_STATE)
alpha_options = [0.25, 0.5, 1.0, 2.0]
for k in range(N_RANDOM_BLENDS):
    alpha = alpha_options[k % len(alpha_options)]
    w = rng.dirichlet(np.ones(len(model_names)) * alpha)
    score, diag, _, _ = score_blend(w)
    if k < 25 or score < best['score']:
        blend_rows.append({'blend': f'random_{k}_a{alpha}', 'AW-MAE': score, **{f'w_{n}': w[j] for j, n in enumerate(model_names)}})
    if score < best['score']:
        best = {'score': score, 'weights': w.copy(), 'name': f'random_{k}_a{alpha}', 'diag': diag}

blend_df = pd.DataFrame(blend_rows).sort_values('AW-MAE').reset_index(drop=True)
display(blend_df.head(20))

print('Best blend:', best['name'])
print('Best AW-MAE:', best['score'])
print('Diagnostics:', best['diag'])
print('Weights:')
for name, weight in sorted(zip(model_names, best['weights']), key=lambda x: -x[1]):
    print(f'  {name:<24} {weight:.5f}')

Models in bank: ['cat_mae_d7_seed42', 'cat_mae_d6_seed7', 'cat_mae_d8_seed99', 'cat_rmse_d7_seed123', 'lgbm_mae_seed42', 'lgbm_mae_seed99', 'xgb_mae_seed42', 'xgb_mae_seed123']
Prediction matrix: (8, 15755) (8, 15755)


,blend,AW-MAE,w_cat_mae_d7_seed42,w_cat_mae_d6_seed7,w_cat_mae_d8_seed99,w_cat_rmse_d7_seed123,w_lgbm_mae_seed42,w_lgbm_mae_seed99,w_xgb_mae_seed42,w_xgb_mae_seed123
0,random_2048_a0.25,3.105088,0.164635,0.016605,0.064692,0.503541,0.048325,0.168878,0.028847,0.004477
1,random_444_a0.25,3.105457,0.000012,0.057430,0.236373,0.509614,0.010103,0.101200,0.084116,0.001152
2,random_394_a1.0,3.107153,0.213047,0.054711,0.004045,0.450474,0.018895,0.099692,0.100238,0.058898
3,random_1_a0.5,3.107529,0.041741,0.143013,0.099918,0.493764,0.110321,0.003194,0.093807,0.014242
4,random_17_a0.5,3.124107,0.032388,0.370874,0.005844,0.552852,0.010914,0.014448,0.012442,0.000236
5,random_22_a1.0,3.135507,0.138973,0.109136,0.025253,0.331335,0.004094,0.118775,0.010287,0.262147
6,random_12_a0.25,3.136100,0.506175,0.000001,0.112283,0.307730,0.003562,0.068524,0.000028,0.001696
7,random_3_a2.0,3.141638,0.044527,0.074500,0.143376,0.241109,0.212933,0.162781,0.031832,0.088941
8,random_21_a0.5,3.143283,0.026908,0.028862,0.017666,0.389485,0.021612,0.241303,0.188301,0.085865
9,single_cat_rmse_d7_seed123,3.149484,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000


Best blend: random_2048_a0.25
Best AW-MAE: 3.1050876553121327
Diagnostics: {'AW-MAE': np.float64(3.1050876553121327), 'MAE_raw_component': np.float64(1.017676927959378), 'exact_acc': np.float64(0.09946048873373532), 'outcome_acc': np.float64(0.5245318946366233), 'goal_diff_acc': np.float64(0.23973341796255157)}
Weights:
  cat_rmse_d7_seed123      0.50354
  lgbm_mae_seed99          0.16888
  cat_mae_d7_seed42        0.16464
  cat_mae_d8_seed99        0.06469
  lgbm_mae_seed42          0.04832
  xgb_mae_seed42           0.02885
  cat_mae_d6_seed7         0.01661
  xgb_mae_seed123          0.00448


In [14]:
clip_rows = []
for max_score in range(4, 9):
    score, diag, _, _ = score_blend(best['weights'], max_score=max_score)
    clip_rows.append({'max_score': max_score, **diag})
clip_df = pd.DataFrame(clip_rows).sort_values('AW-MAE')
display(clip_df)
BEST_MAX_SCORE = int(clip_df.iloc[0]['max_score'])
print('BEST_MAX_SCORE:', BEST_MAX_SCORE)

,max_score,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
4,8,3.104763,1.017455,0.099651,0.524532,0.239924
3,7,3.104795,1.017518,0.099524,0.524532,0.239797
2,6,3.105088,1.017677,0.099460,0.524532,0.239733
1,5,3.107540,1.018724,0.098762,0.524532,0.239416
0,4,3.113419,1.020914,0.099841,0.524532,0.240939


BEST_MAX_SCORE: 8


In [19]:
final_pred_bank = {}
final_models = {}

# CatBoost final models
for cfg in CATBOOST_CONFIGS:
    name = cfg['name']
    if name not in model_names:
        continue

    val_t, val_o = trained_val_models[name]
    best_iter = int(max(getattr(val_t, 'best_iteration_', 800), getattr(val_o, 'best_iteration_', 800)) + 120)
    params = {**BASE_CAT_PARAMS, **cfg['params']}
    params['iterations'] = max(best_iter, 300)
    params.pop('eval_metric', None)

    print('Final training', name, 'iterations:', params['iterations'])
    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_full, y_full_team, cat_features=cat_feature_indices, verbose=150)
    mo.fit(X_full, y_full_opp, cat_features=cat_feature_indices, verbose=150)

    pred_t = np.clip(mt.predict(X_test), 0, None)
    pred_o = np.clip(mo.predict(X_test), 0, None)
    final_pred_bank[name] = (pred_t, pred_o)
    final_models[name] = (mt, mo)

# LightGBM final models, only if they were used in validation.
if 'HAS_LGBM' in globals() and HAS_LGBM:
    for name, params in [
        ('lgbm_mae_seed42', dict(objective='regression_l1', n_estimators=1400, learning_rate=0.035, num_leaves=63, subsample=0.85, colsample_bytree=0.85, reg_lambda=6.0, random_state=42)),
        ('lgbm_mae_seed99', dict(objective='regression_l1', n_estimators=1200, learning_rate=0.040, num_leaves=47, subsample=0.90, colsample_bytree=0.80, reg_lambda=9.0, random_state=99)),
    ]:
        if name not in model_names:
            continue
        print('Final training', name)
        mt = LGBMRegressor(**params, verbosity=-1)
        mo = LGBMRegressor(**params, verbosity=-1)
        mt.fit(Xfull_enc, y_full_team)
        mo.fit(Xfull_enc, y_full_opp)
        final_pred_bank[name] = (np.clip(mt.predict(Xtest_enc), 0, None), np.clip(mo.predict(Xtest_enc), 0, None))
        final_models[name] = (mt, mo)

# XGBoost final models, only if they were used in validation.
if 'HAS_XGB' in globals() and HAS_XGB:
    for name, params in [
        ('xgb_mae_seed42', dict(n_estimators=1200, learning_rate=0.035, max_depth=6, min_child_weight=8, subsample=0.85, colsample_bytree=0.85, reg_lambda=8.0, random_state=42)),
        ('xgb_mae_seed123', dict(n_estimators=1000, learning_rate=0.040, max_depth=5, min_child_weight=10, subsample=0.90, colsample_bytree=0.80, reg_lambda=10.0, random_state=123)),
        ]:
        if name not in model_names:
            continue
        print('Final training', name)
        base = dict(objective='reg:absoluteerror', tree_method='hist', n_jobs=-1)
        mt = XGBRegressor(**base, **params)
        mo = XGBRegressor(**base, **params)
        mt.fit(Xfull_enc, y_full_team, verbose=False)
        mo.fit(Xfull_enc, y_full_opp, verbose=False)
        final_pred_bank[name] = (np.clip(mt.predict(Xtest_enc), 0, None), np.clip(mo.predict(Xtest_enc), 0, None))
        final_models[name] = (mt, mo)

print('Final prediction bank:', list(final_pred_bank.keys()))

Final training cat_mae_d7_seed42 iterations: 1519


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1667505	total: 16.9ms	remaining: 25.6s
150:	learn: 1.1031283	total: 2.77s	remaining: 25.1s
300:	learn: 1.0756391	total: 5.56s	remaining: 22.5s
450:	learn: 1.0573632	total: 8.96s	remaining: 21.2s
600:	learn: 1.0445851	total: 13.2s	remaining: 20.2s
750:	learn: 1.0358789	total: 17.1s	remaining: 17.5s
900:	learn: 1.0296185	total: 20s	remaining: 13.7s
1050:	learn: 1.0244838	total: 22.7s	remaining: 10.1s
1200:	learn: 1.0195425	total: 25.5s	remaining: 6.74s
1350:	learn: 1.0153143	total: 28.3s	remaining: 3.51s
1500:	learn: 1.0118433	total: 32.3s	remaining: 387ms
1518:	learn: 1.0113716	total: 32.8s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1668032	total: 24.8ms	remaining: 37.7s
150:	learn: 1.1027620	total: 3.72s	remaining: 33.7s
300:	learn: 1.0746607	total: 7.64s	remaining: 30.9s
450:	learn: 1.0566064	total: 11.5s	remaining: 27.2s
600:	learn: 1.0462952	total: 15.5s	remaining: 23.6s
750:	learn: 1.0382323	total: 19.5s	remaining: 20s
900:	learn: 1.0308700	total: 23.6s	remaining: 16.2s
1050:	learn: 1.0244873	total: 27.8s	remaining: 12.4s
1200:	learn: 1.0194762	total: 31.7s	remaining: 8.38s
1350:	learn: 1.0151241	total: 35.3s	remaining: 4.38s
1500:	learn: 1.0115429	total: 38.9s	remaining: 467ms
1518:	learn: 1.0110457	total: 39.4s	remaining: 0us
Final training cat_mae_d6_seed7 iterations: 1719


...

0:	learn: 1.1667868	total: 25.1ms	remaining: 43.1s
150:	learn: 1.1124424	total: 3.44s	remaining: 35.7s
300:	learn: 1.0882276	total: 7.07s	remaining: 33.3s
450:	learn: 1.0710002	total: 10.7s	remaining: 30.1s
600:	learn: 1.0598835	total: 14.3s	remaining: 26.7s
750:	learn: 1.0521691	total: 18s	remaining: 23.2s
900:	learn: 1.0465718	total: 21.5s	remaining: 19.5s
1050:	learn: 1.0420180	total: 25.2s	remaining: 16s
1200:	learn: 1.0381032	total: 28.9s	remaining: 12.4s
1350:	learn: 1.0344812	total: 32.2s	remaining: 8.78s
1500:	learn: 1.0311723	total: 34.7s	remaining: 5.04s
1650:	learn: 1.0282211	total: 37.2s	remaining: 1.53s
1718:	learn: 1.0268312	total: 38.3s	remaining: 0us
Final training cat_mae_d8_seed99 iterations: 1319


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1665961	total: 19.1ms	remaining: 25.2s
150:	learn: 1.1072112	total: 3.35s	remaining: 25.9s
300:	learn: 1.0793209	total: 6.56s	remaining: 22.2s
450:	learn: 1.0590443	total: 9.85s	remaining: 19s
600:	learn: 1.0457144	total: 13.2s	remaining: 15.8s
750:	learn: 1.0345075	total: 16.5s	remaining: 12.5s
900:	learn: 1.0274196	total: 19.8s	remaining: 9.19s
1050:	learn: 1.0216845	total: 23.1s	remaining: 5.89s
1200:	learn: 1.0162728	total: 26.3s	remaining: 2.59s
1318:	learn: 1.0121818	total: 28.8s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1665441	total: 21.1ms	remaining: 27.8s
150:	learn: 1.1075924	total: 3.34s	remaining: 25.8s
300:	learn: 1.0792949	total: 6.67s	remaining: 22.6s
450:	learn: 1.0592805	total: 10.4s	remaining: 20s
600:	learn: 1.0462969	total: 13.6s	remaining: 16.3s
750:	learn: 1.0369641	total: 18s	remaining: 13.6s
900:	learn: 1.0279716	total: 22.7s	remaining: 10.5s
1050:	learn: 1.0209883	total: 26.5s	remaining: 6.76s
1200:	learn: 1.0155362	total: 29.8s	remaining: 2.93s
1318:	learn: 1.0117281	total: 32.4s	remaining: 0us
Final training cat_rmse_d7_seed123 iterations: 950
0:	learn: 1.7803131	total: 24.7ms	remaining: 23.4s
150:	learn: 1.5027643	total: 2.72s	remaining: 14.4s
300:	learn: 1.4632133	total: 5.35s	remaining: 11.5s
450:	learn: 1.4382194	total: 8.05s	remaining: 8.9s
600:	learn: 1.4223619	total: 10.7s	remaining: 6.22s
750:	learn: 1.4078502	total: 13.3s	remaining: 3.54s
900:	learn: 1.3946756	total: 16s	remaining: 872ms
949:	learn: 1.3908281	total: 16.9s	remaining: 0us
0:	learn: 1.7804701	tot

In [ ]:
# Align final predictions with validation model order.
missing_final = [name for name in model_names if name not in final_pred_bank]
if missing_final:
    raise ValueError(f'Missing final predictions for: {missing_final}')

test_team_matrix = np.vstack([final_pred_bank[name][0] for name in model_names])
test_opp_matrix = np.vstack([final_pred_bank[name][1] for name in model_names])
weights = best['weights'] / best['weights'].sum()

test_pred_team_raw = np.average(test_team_matrix, axis=0, weights=weights)
test_pred_opp_raw = np.average(test_opp_matrix, axis=0, weights=weights)

# Backup: original round+clip ensemble.
round_team_int, round_opp_int = postprocess_round_clip(test_pred_team_raw, test_pred_opp_raw, max_score=BEST_MAX_SCORE)
submission_round = sample[['Id']].copy()
submission_round['team_goals'] = round_team_int
submission_round['opp_goals'] = round_opp_int
submission_round.to_csv(SUBMISSION_ROUNDCLIP_PATH, index=False)

# Jalur A: outcome-aware post-processing from validation-calibrated params.
if 'BEST_PP_PARAMS' in globals() and BEST_PP_PARAMS.get('mode') == 'outcome_aware':
    full_prior_matrix = build_score_pair_prior(train_prep, max_score=int(BEST_PP_PARAMS['max_score']), smoothing=1.0)
    final_team_int, final_opp_int = outcome_aware_postprocess(
        test_pred_team_raw,
        test_pred_opp_raw,
        BEST_PP_PARAMS,
        prior_matrix=full_prior_matrix,
    )
    active_mode = 'outcome_aware'
else:
    final_team_int, final_opp_int = round_team_int, round_opp_int
    active_mode = 'round_clip'

submission = sample[['Id']].copy()
submission['team_goals'] = final_team_int
submission['opp_goals'] = final_opp_int

assert submission.shape == sample.shape
assert submission['Id'].equals(sample['Id'])
submission.to_csv(SUBMISSION_PATH, index=False)

changed_rows = ((submission['team_goals'] != submission_round['team_goals']) | (submission['opp_goals'] != submission_round['opp_goals'])).sum()

print('Saved Jalur A submission:', SUBMISSION_PATH)
print('Saved round+clip backup:', SUBMISSION_ROUNDCLIP_PATH)
print('Active postprocess mode:', active_mode)
print('Best validation blend AW-MAE:', best['score'])
if 'BEST_PP_SCORE' in globals():
    print('Best validation Jalur A AW-MAE:', BEST_PP_SCORE)
if 'BEST_PP_PARAMS' in globals():
    print('Best postprocess params:', BEST_PP_PARAMS)
print('Changed rows vs round+clip:', changed_rows)
print('Submission shape:', submission.shape)

display(submission.head())
display(submission[['team_goals', 'opp_goals']].describe())

print('
distribusi skor')
print('
team_goals')
print(submission['team_goals'].value_counts())
print('
opp_goals')
print(submission['opp_goals'].value_counts())

print('
distribusi pasangan skor')
display(submission[['team_goals', 'opp_goals']].value_counts().head(30).to_frame('count'))



kemungkinan bottleneck-nya bukan model type, tapi fitur. Upgrade berikutnya yang paling bernilai:
1. reconstruct historical features untuk test,
2. outcome-aware post-processing,
3. separate classifier untuk win/draw/loss,
4. calibration skor integer dari validation.